# 02. 環境を触って理解する（ローカルのみ）

**対応するテキスト**: [docs/01_模倣学習の基礎.md](../docs/01_模倣学習の基礎.md) の 1.5

> [!NOTE]
> **このノートブックは手元の PC だけで完結します。Azure の課金は発生しません。**
> 実行時間も短く、何度でもやり直せます。

ここで確認すること:

1. ロボットの**観測（25 個の数値）と行動（4 つの数値）**
2. ⭐ **素の環境は「成功した瞬間に終わる」**（可変ホライズン）
3. ⭐ **固定ホライズン化すると、成績を変えずにエピソード長だけ揃う**
4. **⚠ 吸収状態では `info` が空になり、成功情報が消える**
5. 一様ランダム行動の成績（正規化リターンの 0 点）

> ⚠ **なぜこれを最初に確認するのか**
> [01 章 1.5](../docs/01_模倣学習の基礎.md) のとおり、**エピソード長が変わる環境では GAIL / AIRL の評価が意味を失います。**
> この事実を「読んで納得する」のではなく、**自分の目で測って確認する**のがこのノートブックの目的です。

In [ ]:
# ../src のスクリプトを import できるようにする（本ハンズオンの実装をそのまま使うため）
import sys

sys.path.insert(0, "../src")

from importlib import metadata

import gymnasium as gym
import numpy as np

from pick_place_env import (
    BASE_ENV_ID,
    DEFAULT_ENV_ID,
    HORIZON,
    VARIABLE_HORIZON_ENV_ID,
    unflatten_observation,
)
from scripted_expert import scripted_action

for name in ("numpy", "gymnasium", "panda-gym", "pybullet", "stable-baselines3", "imitation", "seals"):
    print(f"  {name:<20} {metadata.version(name)}")

print()
print("素の環境       :", BASE_ENV_ID)
print("固定ホライズン :", DEFAULT_ENV_ID)
print("1 エピソード   :", HORIZON, "ステップ")

## 1. 観測と行動を見る

**観測**（observation）= いま何が起きているか。**行動**（action）= 何ができるか。

本ハンズオンの環境は、観測を **1 本の 25 次元ベクトル**に平坦化しています。
元は 3 つのキーを持つ辞書なので、`unflatten_observation()` で戻せます。

In [ ]:
env = gym.make(DEFAULT_ENV_ID)
try:
    print("observation_space :", env.observation_space)
    print("action_space      :", env.action_space)
    print()

    obs, info = env.reset(seed=0)
    parts = unflatten_observation(obs)
    for key in sorted(parts):
        print(f"  {key:<14} shape={np.asarray(parts[key]).shape}  {np.round(parts[key], 3)}")

    print()
    print("観測の読み方（実測で確認済み）")
    print("  observation[0:3]  = 手先（エンドエフェクター）の位置")
    print("  observation[3:6]  = 手先の速度")
    print("  observation[6]    = 指の開き幅（全開 0.08 / 全閉 0.0）")
    print("  observation[7:10] = 物体の位置（achieved_goal と一致する）")
    print("  achieved_goal     = いまの物体の位置")
    print("  desired_goal      = 物体を運ぶ先")
    print()
    print("行動の読み方")
    print("  action[0:3] = 手先を x / y / z に動かす量（× 0.05 m）")
    print("  action[3]   = 指の開閉（正で開く / 負で閉じる。× 0.2 を現在幅に加算）")
finally:
    env.close()

## 2. ⭐ 素の環境は「成功した瞬間に終わる」

[../src/scripted_expert.py](../src/scripted_expert.py) の**スクリプト専門家**で動かします。
専門家はほぼ必ず成功するので、**終了条件の違いがはっきり出ます。**

> ⚠ **ランダム行動では違いが見えません。** ランダムはまず成功しないので、
> どちらの環境でも 50 ステップで打ち切られるだけです（4. で確認します）。

In [ ]:
def run_episodes(env_id, action_fn, n_episodes=5, seed=0):
    env = gym.make(env_id)
    rows = []
    try:
        for episode in range(n_episodes):
            obs, _ = env.reset(seed=seed + episode)
            steps, total, success = 0, 0.0, False
            while True:
                obs, reward, terminated, truncated, info = env.step(action_fn(obs))
                steps += 1
                total += float(reward)
                success = success or bool(info.get("is_success", False))
                if terminated or truncated:
                    break
            rows.append(
                {"episode": episode, "steps": steps, "return": total,
                 "success": success, "terminated": bool(terminated), "truncated": bool(truncated)}
            )
    finally:
        env.close()
    return rows


import pandas as pd

raw_rows = run_episodes(VARIABLE_HORIZON_ENV_ID, scripted_action)
print("素の環境（可変ホライズン）")
pd.DataFrame(raw_rows)

## 3. ⭐ 固定ホライズン化した環境

同じ専門家を、`AbsorbAfterDoneWrapper` で固定ホライズン化した環境で動かします。

In [ ]:
fixed_rows = run_episodes(DEFAULT_ENV_ID, scripted_action)
print("固定ホライズン化した環境")
pd.DataFrame(fixed_rows)

In [ ]:
print("=== 比較 ===")
print(f"{'ep':>3}  {'素:steps':>9} {'素:return':>10}  {'固定:steps':>11} {'固定:return':>12}")
for a, b in zip(raw_rows, fixed_rows):
    print(f"{a['episode']:>3}  {a['steps']:>9} {a['return']:>10.1f}  {b['steps']:>11} {b['return']:>12.1f}")

same_return = all(abs(a["return"] - b["return"]) < 1e-9 for a, b in zip(raw_rows, fixed_rows))
fixed_lengths = {b["steps"] for b in fixed_rows}
print()
print("固定側のエピソード長 :", fixed_lengths, "← 1 種類だけなら固定ホライズン化できています")
print("リターンは一致したか :", same_return, "← 成績の意味は変わっていません")

### 何が起きたのか

```mermaid
flowchart LR
    A["素の環境<br/>成功で terminated=True"] --> B["AbsorbAfterDoneWrapper<br/>終了せず「何も起きない状態」を続ける<br/>報酬は 0"]
    B --> C["TimeLimit(50)<br/>必ず 50 ステップで truncated"]
```

- **リターンは変わりません。** 成功後の報酬を 0 にしているので、
  「何ステップ目で成功したか」という成績の意味はそのままです。
- **エピソード長から成功が読み取れなくなりました。** これが目的です。

> この構成は、`imitation` の姉妹プロジェクト **seals** が `MountainCar` に対して行っているものと同じです。
> 出典（参考情報・OSS 公式ソース）: https://github.com/HumanCompatibleAI/seals/blob/master/src/seals/classic_control.py

> **なぜ必要なのか**は [01 章 1.5](../docs/01_模倣学習の基礎.md) と [07 章](../docs/07_GAILと評価の落とし穴.md) にあります。

## 4. ⚠ 吸収状態では `info` が空になる（落とし穴）

`AbsorbAfterDoneWrapper` のソースには、吸収状態について次のように書かれています。

> "In this artificial absorb state, we stop calling `self.env.step(action)` ... **`info` is always an empty dictionary.**"
>
> 出典（参考情報・OSS 公式ソース）: https://github.com/HumanCompatibleAI/seals/blob/master/src/seals/util.py

**panda-gym は成功を `info["is_success"]` で返します。**
つまり **そのままでは、成功した次のステップから成功情報が消えます。**

[../src/pick_place_env.py](../src/pick_place_env.py) の `SuccessRecorderWrapper` がこれを保持しています。
下のセルは、**そのラッパーを外すとどうなるか**を確かめます。

In [ ]:
from gymnasium.wrappers import FlattenObservation, TimeLimit
from seals.util import AbsorbAfterDoneWrapper

from pick_place_env import ClipObservationWrapper

# SuccessRecorderWrapper を「入れない」構成
no_recorder = TimeLimit(
    ClipObservationWrapper(FlattenObservation(AbsorbAfterDoneWrapper(gym.make(BASE_ENV_ID)))),
    max_episode_steps=HORIZON,
)

obs, _ = no_recorder.reset(seed=0)
steps_with_success, empty_info_steps = [], []
for step in range(HORIZON):
    obs, _, terminated, truncated, info = no_recorder.step(scripted_action(obs))
    if info.get("is_success"):
        steps_with_success.append(step)
    if info == {}:
        empty_info_steps.append(step)
    if terminated or truncated:
        break
no_recorder.close()

print("is_success が True だったステップ :", steps_with_success)
print("info が空だったステップ数         :", len(empty_info_steps))
print("最初に info が空になったステップ   :", empty_info_steps[0] if empty_info_steps else None)
print()
print("→ 成功を報告するのは 1 ステップだけ。その後は info が空になり、成功が分からなくなります。")

## 5. 一様ランダム行動の成績（正規化リターンの 0 点）

「ランダムに動かしたときの点数」が、**正規化リターンの 0 点**になります。

$$
\text{normalized\_return} = \frac{\text{方策のリターン} - \text{ランダムのリターン}}{\text{専門家のリターン} - \text{ランダムのリターン}}
$$

In [ ]:
random_env = gym.make(DEFAULT_ENV_ID)
random_env.action_space.seed(0)
random_rows = run_episodes(DEFAULT_ENV_ID, lambda obs: random_env.action_space.sample())
random_env.close()

df = pd.DataFrame(random_rows)
print("一様ランダム")
print(f"  平均リターン : {df['return'].mean():.2f}")
print(f"  成功率       : {df['success'].mean():.2f}")
print()
print("→ ランダムでは成功しないため、素の環境でも 50 ステップで打ち切られます。")
print("　 2. の違いが見えたのは、専門家が実際に成功したからです。")
df

## 6.【任意】GUI でロボットを見る

`render_mode="human"` にすると、PyBullet の **OpenGL ウィンドウが別ウィンドウとして開きます。**

> [!WARNING]
> - **GUI ウィンドウはノートブックの中ではなく、別ウィンドウとして開きます。** 画面の裏に隠れていないか確認してください。
> - **`env.close()` を必ず実行してください。** 実行しないとウィンドウが残ります。
> - **リモート デスクトップや仮想マシンでは OpenGL が使えず、起動に失敗することがあります。**
> - **Azure ML のコンピューティングでは使えません**（ヘッドレスのため）。ローカル専用です。

In [ ]:
# ============================================================
#  GUI の動作確認（手元の PC でのみ動きます）
#  ※ 別ウィンドウが開きます。専門家がピックアンドプレースを実演して自動的に閉じます。
# ============================================================
import time

gui = gym.make(DEFAULT_ENV_ID, render_mode="human")
try:
    obs, _ = gui.reset(seed=0)
    for _ in range(HORIZON):
        obs, _, terminated, truncated, _ = gui.step(scripted_action(obs))
        time.sleep(0.03)          # 目で追えるように少し遅くする
        if terminated or truncated:
            break
    time.sleep(1.0)
finally:
    gui.close()                    # ウィンドウを必ず閉じる

print("OK: GUI が起動し、正常に終了しました。")

## 7. まとめ

- 観測は **25 次元のベクトル**、行動は **4 次元の連続値**
- **素の環境は成功した瞬間に終わる**（可変ホライズン）
- **固定ホライズン化しても成績（リターン）は変わらない。** 変わるのはエピソード長だけ
- **吸収状態では `info` が空になる**ので、成功情報を保持するラッパーが要る
- ランダムでは成功しないため、**ランダムだけを見ていると可変ホライズンの問題に気づけない**

## ✅ チェックリスト

- [ ] 観測と行動の形を確認した
- [ ] **素の環境でエピソード長がばらつく**ことを確認した
- [ ] **固定ホライズン化するとエピソード長が 1 種類になる**ことを確認した
- [ ] **リターンが両者で一致する**ことを確認した
- [ ] **`info` が空になる落とし穴**を確認した
- [ ] 一様ランダムの成績を測った

> ⚠ **このノートブックを実行すると、カレント フォルダーに `mlruns/` と `mlflow.db` が作られることがあります。**
> `.gitignore` で除外済みなのでコミットされませんが、不要なら削除して構いません。

---

**次へ**: [docs/03_AzureML環境構築.md](../docs/03_AzureML環境構築.md) → [01_setup_azureml.ipynb](01_setup_azureml.ipynb)